In [0]:
data = [
    ("C001", 500, "2024-01-05"),
    ("C001", 200, "2024-01-01"),
    ("C002", 150, "2024-01-03"),
    ("C001", 700, "2024-01-10"),
    ("C002", 300, "2024-01-02")
]
columns = ["customer_id", "txn_amount", "txn_date"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec = Window.partitionBy('customer_id').orderBy('txn_date')

df_min_max = df.groupBy("customer_id").agg(min("txn_date").alias('min_date'),
                                   max('txn_date').alias('max_date'))

display(df_min_max)


In [0]:
df_a = df.alias("a")
df_b = df_min_max.alias("b")
df_res = df_a.join(
    df_b,
    (df_a["customer_id"] == df_b["customer_id"]) & ((df_a["txn_date"] == df_b["min_date"]) | (df_a["txn_date"] == df_b["max_date"])),
    "inner"
).select(df_a["customer_id"], df_a["txn_date"], df_a["txn_amount"])
display(df_res)